In [36]:
import os
os.chdir('/home/pedro_alves_coutinho_alumni_usp_/gitworkspace/split_offdb')
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString, Polygon, MultiPolygon, LinearRing
from shapely.ops import split, linemerge, polygonize
import pandas as pd
import logging
import json
import time
import shapely as shp
from multiprocessing import Pool
from functools import partial
import numpy as np
from rtree import index
import psutil
from shapely.strtree import STRtree
from sqlalchemy import text, create_engine
from dotenv import load_dotenv
from sqlalchemy import Table, MetaData, Index
from split import Splitter
from sqlalchemy.dialects.postgresql import ARRAY, TEXT, INTEGER, NUMERIC, BIGINT

# Carregar variáveis do .env para conexão com o banco
load_dotenv()
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")


engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)
engine

start_time = time.time()

with open("config.json", "r") as f:
    config = json.load(f)  


s = Splitter()
s.colunas_boleanas(engine)
s.create_table(engine)

In [57]:
s._intersection_sql(n_grid = 500, engine = engine)
s.prepare_split_line()
s.perform_split()
s.process_overlapping()

'0.49'

In [58]:

# 1. Dropa coluna id
s.gdf_broken_glass.drop(columns='id', inplace=True)       

# 2. Drop onde é id_layer = ['GRID'] através da condiução hexadecimal = 0. O objetivo é descartar geometrias que não tem nenhuma informação
s.gdf_broken_glass=s.gdf_broken_glass[s.gdf_broken_glass['hexadecimal'] != 0]     
s.gdf_broken_glass 


,geometry,id_layer,id_feature,hexadecimal
0,"POLYGON ((-63 -3.95956, -62.99794 -3.95956, -6...","[GRID, MUN, CAR, BIOMA]","[500, 1301209, 526143, 1]",1101
1,"POLYGON ((-62.99794 -3.95956, -63 -3.95956, -6...","[GRID, CAR, MUN, BIOMA]","[500, 526143, 1301209, 1]",1101
2,"POLYGON ((-62.99794 -3.95956, -62.99771 -3.959...","[GRID, MUN, BIOMA]","[500, 1301209, 1]",1100
3,"POLYGON ((-62.99771 -3.95956, -62.99794 -3.959...","[GRID, MUN, BIOMA]","[500, 1301209, 1]",1100
4,"POLYGON ((-62.99771 -3.95956, -62.99738 -3.959...","[GRID, MUN, CAR, BIOMA]","[500, 1301209, 542034, 1]",1101
...,...,...,...,...
225,"POLYGON ((-62.62943 -3.84935, -62.62951 -3.849...","[GRID, CAR, MUN, BIOMA]","[500, 1007740, 1301209, 1]",1101
226,"POLYGON ((-62.90415 -3.82272, -62.90166 -3.821...","[GRID, CAR, MUN, BIOMA]","[500, 393230, 1301209, 1]",1101
227,"POLYGON ((-62.87607 -3.8282, -62.8658 -3.82628...","[GRID, CAR, MUN, BIOMA]","[500, 665746, 1301209, 1]",1101
228,"POLYGON ((-62.91251 -3.86496, -62.91252 -3.864...","[GRID, CAR, MUN, BIOMA]","[500, 744642, 1301209, 1]",1101


In [59]:

for col in s.value_columns:
        #Adiciona colunas de s.value_columns + cd_uf quando houver 'MUN'
        #Define a mask, que é quando existe a col na array, para nao dar warning
            
            coluna=f"cd_{col.lower()}"

            mask = s.gdf_broken_glass['id_layer'].apply(lambda xs: col in xs)
            s.gdf_broken_glass[coluna] = pd.Series([pd.NA] * len(s.gdf_broken_glass), dtype="Int64")
          
                
            if col == 'mun':
                s.gdf_broken_glass['cd_uf'] = pd.Series([pd.NA] * len(s.gdf_broken_glass), dtype="Int64")

            
            s.gdf_broken_glass.loc[mask, coluna] = s.gdf_broken_glass.loc[mask].apply(
                lambda row: int(row['id_feature'][next(i for i, v in enumerate(row['id_layer']) if str(v).upper() == col)]),
                axis=1
            ).pipe(pd.to_numeric, errors='coerce').astype('Int64')


            if col == 'MUN':
                print('input uf')
                #cd_uf apenas onde há 'MUN'. Isso evita warnings desnecessários.
                s.gdf_broken_glass.loc[mask,'cd_uf'] = (s.gdf_broken_glass.loc[mask, coluna]          
                    .astype(str).str[:2]
                    .astype('int')
                ).where(mask, other=pd.NA).pipe(pd.to_numeric, errors='coerce').astype('Int64')

print(s.gdf_broken_glass)
            


input uf
                                              geometry  \
0    POLYGON ((-63 -3.95956, -62.99794 -3.95956, -6...   
1    POLYGON ((-62.99794 -3.95956, -63 -3.95956, -6...   
2    POLYGON ((-62.99794 -3.95956, -62.99771 -3.959...   
3    POLYGON ((-62.99771 -3.95956, -62.99794 -3.959...   
4    POLYGON ((-62.99771 -3.95956, -62.99738 -3.959...   
..                                                 ...   
225  POLYGON ((-62.62943 -3.84935, -62.62951 -3.849...   
226  POLYGON ((-62.90415 -3.82272, -62.90166 -3.821...   
227  POLYGON ((-62.87607 -3.8282, -62.8658 -3.82628...   
228  POLYGON ((-62.91251 -3.86496, -62.91252 -3.864...   
229  POLYGON ((-62.9148 -3.83038, -62.91273 -3.8289...   

                    id_layer                  id_feature  hexadecimal  \
0    [GRID, MUN, CAR, BIOMA]   [500, 1301209, 526143, 1]         1101   
1    [GRID, CAR, MUN, BIOMA]   [500, 526143, 1301209, 1]         1101   
2         [GRID, MUN, BIOMA]           [500, 1301209, 1]         1100   
3 

In [60]:
# 4. Conta número de CARs na feição
s.gdf_broken_glass['n_car'] = np.array([x.count('CAR') for x in s.gdf_broken_glass['id_layer']])

# 5. Adiciona colunas boleanas
for coluna in s.boleanas:
    alvo = coluna.upper()
    s.gdf_broken_glass[f'is_{coluna.lower()}'] = \
        s.gdf_broken_glass['id_layer'].apply(lambda xs: alvo in xs)   

# 6. Calcula área das feicoes. Para isso é necessario descobrir a zona do grid e reprojar e dado de acordo com a feicao
#Descobre em qual zona está o grid
xmin, ymin, xmax, ymax = s.unidade_split.bounds
longitude_media_grid=(xmin + xmax) / 2
zona = int((longitude_media_grid + 180) / 6) + 1
#Seleciona o epsg correspodente a zona
projecao = s.utm_epsg_brazil[zona]
#Reprojeta apenas para calcular area. No entanto a feicao no banco estará em 4674.
gdf_proj = s.gdf_broken_glass.to_crs(epsg=projecao)
gdf_proj['area_ha'] = gdf_proj.geometry.area/10000
#Inputa area na tabela
s.gdf_broken_glass['area_ha']=gdf_proj['area_ha']

# 7. Cria a coluna id_layer_unico 
s.gdf_broken_glass['id_layer_unico'] = s.gdf_broken_glass['id_layer'].apply(lambda x: np.unique(np.sort(x)))

# 8. Converte as arrays nativas de python para uma string compreensivel pelo db
s.gdf_broken_glass['id_layer'] = s.gdf_broken_glass['id_layer'].apply(lambda x: '{' + ','.join(map(str, x)) + '}')
s.gdf_broken_glass['id_layer_unico'] = s.gdf_broken_glass['id_layer_unico'].apply(lambda x: '{' + ','.join(map(str, x)) + '}')
s.gdf_broken_glass['id_feature'] = s.gdf_broken_glass['id_feature'].apply(lambda x: '{' + ','.join(map(str, x)) + '}')

# 9. Faz o join com a tabela de categorias fundiárias
tabela_indice=pd.read_csv(s.path_visao_fundiaria, sep = ';')            
s.gdf_broken_glass=s.gdf_broken_glass.merge(tabela_indice, on=['hexadecimal',f'{s.coluna_hexadecimal}'], how = 'left')


In [61]:
s.gdf_broken_glass

,geometry,id_layer,id_feature,hexadecimal,cd_mun,cd_uf,cd_bioma,n_car,is_bioma,is_car,is_mun,area_ha,id_layer_unico,label
0,"POLYGON ((-63 -3.95956, -62.99794 -3.95956, -6...","{GRID,MUN,CAR,BIOMA}","{500,1301209,526143,1}",1101,1301209,13,1,1,True,True,True,7.595410,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
1,"POLYGON ((-62.99794 -3.95956, -63 -3.95956, -6...","{GRID,CAR,MUN,BIOMA}","{500,526143,1301209,1}",1101,1301209,13,1,1,True,True,True,0.687619,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
2,"POLYGON ((-62.99794 -3.95956, -62.99771 -3.959...","{GRID,MUN,BIOMA}","{500,1301209,1}",1100,1301209,13,1,0,True,False,True,10756.819072,"{BIOMA,GRID,MUN}",NaN
3,"POLYGON ((-62.99771 -3.95956, -62.99794 -3.959...","{GRID,MUN,BIOMA}","{500,1301209,1}",1100,1301209,13,1,0,True,False,True,0.173812,"{BIOMA,GRID,MUN}",NaN
4,"POLYGON ((-62.99771 -3.95956, -62.99738 -3.959...","{GRID,MUN,CAR,BIOMA}","{500,1301209,542034,1}",1101,1301209,13,1,1,True,True,True,0.014113,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,"POLYGON ((-62.62943 -3.84935, -62.62951 -3.849...","{GRID,CAR,MUN,BIOMA}","{500,1007740,1301209,1}",1101,1301209,13,1,1,True,True,True,10.018512,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
226,"POLYGON ((-62.90415 -3.82272, -62.90166 -3.821...","{GRID,CAR,MUN,BIOMA}","{500,393230,1301209,1}",1101,1301209,13,1,1,True,True,True,118.244296,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
227,"POLYGON ((-62.87607 -3.8282, -62.8658 -3.82628...","{GRID,CAR,MUN,BIOMA}","{500,665746,1301209,1}",1101,1301209,13,1,1,True,True,True,167.413491,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA
228,"POLYGON ((-62.91251 -3.86496, -62.91252 -3.864...","{GRID,CAR,MUN,BIOMA}","{500,744642,1301209,1}",1101,1301209,13,1,1,True,True,True,86.479306,"{BIOMA,CAR,GRID,MUN}",PROPOSTA E TEORIA


In [62]:
s.upload_db(engine)

'0.03'